# 입찰메이트 RFP RAG 시스템

RFP(제안요청서) 문서를 분석하여 핵심 정보를 추출하고 Q&A를 수행하는 RAG 시스템

## 목차
1. 환경 설정 및 패키지 설치
2. 데이터 로드 및 전처리
3. 문서 청킹
4. 임베딩 생성 및 벡터 DB 구축
5. Retrieval 테스트
6. RAG Q&A 시스템
7. 성능 평가
8. 실험 비교

## 1. 환경 설정

In [1]:
# 패키지 설치 (최초 1회)
# !pip install -r requirements.txt

In [2]:
import os
import sys
import warnings
warnings.filterwarnings("ignore")

# 프로젝트 루트 설정
PROJECT_ROOT = os.path.dirname(os.path.abspath("__file__"))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# .env 파일에서 API 키 로드 (하드코딩 금지!)
from dotenv import load_dotenv
load_dotenv(os.path.join(PROJECT_ROOT, ".env"))

# API 키 확인
api_key = os.getenv("OPENAI_API_KEY", "")
if not api_key or api_key == "your_api_key_here":
    print("!! .env 파일에 OPENAI_API_KEY를 설정해 주세요 !!")
    print(f"   경로: {os.path.join(PROJECT_ROOT, '.env')}")
else:
    print(f"API 키 로드 완료 (끝 4자리: ...{api_key[-4:]})")

from configs.config import Config
from src.document_loader import DocumentLoader
from src.chunker import chunk_documents
from src.embedder import EmbeddingModel, VectorStore
from src.retriever import Retriever
from src.generator import RAGGenerator
from src.evaluator import RAGEvaluator, EVALUATION_QUESTIONS

import pandas as pd

# 설정 초기화 (시나리오 B: OpenAI API 기반)
config = Config(
    scenario="B",
    openai_embedding_model="text-embedding-3-small",
    openai_chat_model="gpt-5-mini",
    chunk_size=800,
    chunk_overlap=200,
    chunking_method="naive",
    retrieval_top_k=5,
    retrieval_method="similarity",
    temperature=0.1,
    vectordb_type="chroma",
)

print(f"\n시나리오: {config.scenario}")
print(f"임베딩 모델: {config.embedding_model}")
print(f"생성 모델: {config.chat_model}")
print(f"청킹: {config.chunking_method} (size={config.chunk_size}, overlap={config.chunk_overlap})")
print(f"검색: {config.retrieval_method} (top_k={config.retrieval_top_k})")

API 키 로드 완료 (끝 4자리: ...ISwA)

시나리오: B
임베딩 모델: text-embedding-3-small
생성 모델: gpt-5-mini
청킹: naive (size=800, overlap=200)
검색: similarity (top_k=5)


## 2. 데이터 로드 및 전처리

`data/documents/` 디렉토리에 RFP 문서(PDF, HWP)를 넣고, `data/data_list.csv`에 메타데이터를 준비하세요.

In [3]:
# 메타데이터 확인
metadata_path = os.path.join(PROJECT_ROOT, config.metadata_csv)
if os.path.exists(metadata_path):
    metadata_df = pd.read_csv(metadata_path)
    print(f"메타데이터: {len(metadata_df)}건")
    print(f"컬럼: {list(metadata_df.columns)}")
    display(metadata_df.head())
else:
    metadata_df = None
    print(f"메타데이터 파일이 없습니다: {metadata_path}")
    print("data/data_list.csv 파일을 준비해 주세요.")

메타데이터: 100건
컬럼: ['공고 번호', '공고 차수', '사업명', '사업 금액', '발주 기관', '공개 일자', '입찰 참여 시작일', '입찰 참여 마감일', '사업 요약', '파일형식', '파일명', '텍스트']


,공고 번호,공고 차수,사업명,사업 금액,발주 기관,공개 일자,입찰 참여 시작일,입찰 참여 마감일,사업 요약,파일형식,파일명,텍스트
0,20241001798,0.0,한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화,130000000.0,한영대학,2024-10-04 13:51:23,NaN,2024-10-15 17:00:00,- 한영대학교 특성화 맞춤형 교육환경 구축을 위해 트랙운영 학사정보시스템을 고도화한...,hwp,한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp,\n \n2024년 특성화 맞춤형 교육환경 구축 – 트랙운영 학사정보시스템 ...
1,20241002912,0.0,2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선,129300000.0,한국연구재단,2024-10-04 15:01:52,2024-10-14 10:00:00,2024-10-16 14:00:00,- 사업 개요: 2024년 대학 산학협력활동 실태조사 시스템(UICC) 기능개선\n...,hwp,한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선.hwp,\r\n \r\n \r\n \r\n제 안 요 청 서\r\n[ 2024년 대학 ...
2,20240827859,0.0,EIP3.0 고압가스 안전관리 시스템 구축 용역,40000000.0,한국생산기술연구원,2024-08-28 11:31:02,2024-08-29 09:00:00,2024-09-09 10:00:00,- 사업 개요: EIP3.0 고압가스 안전관리 시스템 구축 용역\n- 추진배경: 안...,hwp,한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역.hwp,\r\n \r\nEIP3.0 고압가스 안전관리\r\n시스템 구축 용역\...
3,20240430918,0.0,도시계획위원회 통합관리시스템 구축용역,150000000.0,인천광역시,2024-04-18 16:26:32,2024-05-02 10:00:00,2024-05-09 16:00:00,- 사업명: 도시계획위원회 통합관리시스템 구축 용역\n- 용역개요: 도시계획위원회와...,hwp,인천광역시_도시계획위원회 통합관리시스템 구축용역.hwp,\r\n \r\n \r\n도시계획위원회 통합관리시스템 구축\r\n제 안 요 청...
4,20240430896,0.0,봉화군 재난통합관리시스템 고도화 사업(협상)(긴급),900000000.0,경상북도 봉화군,2024-04-18 16:33:28,2024-04-26 09:00:00,2024-04-30 17:00:00,- 사업명: 봉화군 재난통합관리시스템 고도화 사업\n- 사업개요: 공동수급(공동이행...,hwp,경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급).hwp,\r\n \r\n \r\n제안요청서\r\n \r\n사 업 명\r\n봉화...


In [4]:
# 문서 로드 (PDF, HWP 모두 지원)
loader = DocumentLoader(
    documents_dir=os.path.join(PROJECT_ROOT, config.documents_dir),
    metadata_csv=metadata_path if metadata_df is not None else None,
)

print("문서 로드 중...")
documents = loader.load_all()

# 로드된 문서 통계
if documents:
    total_chars = sum(len(d["text"]) for d in documents)
    avg_chars = total_chars / len(documents)
    print(f"\n총 문자 수: {total_chars:,}")
    print(f"평균 문자 수: {avg_chars:,.0f}")
    
    # 샘플 확인
    print(f"\n첫 번째 문서 미리보기 (처음 500자):")
    print(documents[0]["text"][:500])
    print(f"\n메타데이터: {documents[0]['metadata']}")
else:
    print("로드된 문서가 없습니다. data/documents/ 디렉토리에 PDF/HWP 파일을 넣어주세요.")

문서 로드 중...
  ✓ 로드 완료: (사)벤처기업협회_2024년 벤처확인종합관리시스템 기능 고도화 용역사업 .hwp (100835 chars)
  ✓ 로드 완료: (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp (49576 chars)
  ✓ 로드 완료: (사）한국대학스포츠협의회_KUSF 체육특기자 경기기록 관리시스템 개발.hwp (71338 chars)
  ✓ 로드 완료: (재)예술경영지원센터_통합 정보시스템 구축 사전 컨설팅.hwp (40929 chars)
  ✓ 로드 완료: 2025 구미 아시아육상경기선수권대회 조직위원회_2025 구미아시아육상경.hwp (45538 chars)
  ✓ 로드 완료: BioIN_의료기기산업 종합정보시스템(정보관리기관) 기능개선 사업(2차).hwp (68039 chars)
  ✓ 로드 완료: KOICA 전자조달_[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원 .hwp (124702 chars)
  ✓ 로드 완료: 경기도 안양시_호계체육관 배드민턴장 및 탁구장 예약시스템 구축 용역.hwp (60872 chars)
  ✓ 로드 완료: 경기도 평택시_2024년도 평택시 버스정보시스템(BIS) 구축사업.hwp (77392 chars)
  ✓ 로드 완료: 경기도사회서비스원_2024년 통합사회정보시스템 운영지원.hwp (66902 chars)
  ✓ 로드 완료: 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급).hwp (55848 chars)
  ✓ 로드 완료: 경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp (42703 chars)
  ✓ 로드 완료: 고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf (187536 chars)
  ✓ 로드 완료: 고양도시관리공사_관산근린공원 다목적구장 홈페이지 및 회원 통합운영.hwp (68500 chars)
  ✓ 로드 완료: 광주과학기술원_실시간통합연구비관리시스템(RCMS)  연계 모듈 변경 

## 3. 문서 청킹

두 가지 청킹 방식 비교:
- **Naive**: 고정 크기 기반 (RecursiveCharacterTextSplitter)
- **Semantic**: RFP 문서 섹션 구조를 인식하여 의미 단위 분할

In [5]:
# Naive 청킹
print("=" * 50)
print("Naive 청킹 (chunk_size=800, overlap=200)")
print("=" * 50)
naive_chunks = chunk_documents(
    documents,
    method="naive",
    chunk_size=config.chunk_size,
    chunk_overlap=config.chunk_overlap,
)

chunk_lengths = [len(c["text"]) for c in naive_chunks]
print(f"\n청크 길이 통계:")
print(f"  평균: {sum(chunk_lengths)/len(chunk_lengths):.0f} chars")
print(f"  최소: {min(chunk_lengths)} chars")
print(f"  최대: {max(chunk_lengths)} chars")

Naive 청킹 (chunk_size=800, overlap=200)
  (사)벤처기업협회_2024년 벤처확인종합관리시스템 기능 고도화 용역사업 .hwp: 168 chunks
  (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp: 83 chunks
  (사）한국대학스포츠협의회_KUSF 체육특기자 경기기록 관리시스템 개발.hwp: 118 chunks
  (재)예술경영지원센터_통합 정보시스템 구축 사전 컨설팅.hwp: 69 chunks
  2025 구미 아시아육상경기선수권대회 조직위원회_2025 구미아시아육상경.hwp: 76 chunks
  BioIN_의료기기산업 종합정보시스템(정보관리기관) 기능개선 사업(2차).hwp: 114 chunks
  KOICA 전자조달_[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원 .hwp: 207 chunks
  경기도 안양시_호계체육관 배드민턴장 및 탁구장 예약시스템 구축 용역.hwp: 102 chunks
  경기도 평택시_2024년도 평택시 버스정보시스템(BIS) 구축사업.hwp: 130 chunks
  경기도사회서비스원_2024년 통합사회정보시스템 운영지원.hwp: 113 chunks
  경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급).hwp: 94 chunks
  경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp: 72 chunks
  고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf: 312 chunks
  고양도시관리공사_관산근린공원 다목적구장 홈페이지 및 회원 통합운영.hwp: 115 chunks
  광주과학기술원_실시간통합연구비관리시스템(RCMS)  연계 모듈 변경 사업.hwp: 90 chunks
  광주과학기술원_학사시스템 기능개선 사업.hwp: 73 chunks
  국가과학기술지식정보서비스_통합정보시스템 고도화 용역.hwp: 93 chunks
  국가철도공단_철도인프라 디지털트윈 정보화전략계획(ISP) 수립 용역(

In [6]:
# Semantic 청킹 (비교용)
print("=" * 50)
print("Semantic 청킹 (RFP 구조 인식)")
print("=" * 50)
semantic_chunks = chunk_documents(
    documents,
    method="semantic",
    chunk_size=config.chunk_size,
    chunk_overlap=config.chunk_overlap,
)

sem_lengths = [len(c["text"]) for c in semantic_chunks]
print(f"\n청크 길이 통계:")
print(f"  평균: {sum(sem_lengths)/len(sem_lengths):.0f} chars")
print(f"  최소: {min(sem_lengths)} chars")
print(f"  최대: {max(sem_lengths)} chars")

print(f"\n비교: Naive {len(naive_chunks)}개 vs Semantic {len(semantic_chunks)}개")

Semantic 청킹 (RFP 구조 인식)
  (사)벤처기업협회_2024년 벤처확인종합관리시스템 기능 고도화 용역사업 .hwp: 428 chunks
  (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp: 219 chunks
  (사）한국대학스포츠협의회_KUSF 체육특기자 경기기록 관리시스템 개발.hwp: 328 chunks
  (재)예술경영지원센터_통합 정보시스템 구축 사전 컨설팅.hwp: 160 chunks
  2025 구미 아시아육상경기선수권대회 조직위원회_2025 구미아시아육상경.hwp: 192 chunks
  BioIN_의료기기산업 종합정보시스템(정보관리기관) 기능개선 사업(2차).hwp: 307 chunks
  KOICA 전자조달_[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원 .hwp: 476 chunks
  경기도 안양시_호계체육관 배드민턴장 및 탁구장 예약시스템 구축 용역.hwp: 365 chunks
  경기도 평택시_2024년도 평택시 버스정보시스템(BIS) 구축사업.hwp: 343 chunks
  경기도사회서비스원_2024년 통합사회정보시스템 운영지원.hwp: 330 chunks
  경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급).hwp: 290 chunks
  경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp: 190 chunks
  고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf: 723 chunks
  고양도시관리공사_관산근린공원 다목적구장 홈페이지 및 회원 통합운영.hwp: 385 chunks
  광주과학기술원_실시간통합연구비관리시스템(RCMS)  연계 모듈 변경 사업.hwp: 295 chunks
  광주과학기술원_학사시스템 기능개선 사업.hwp: 247 chunks
  국가과학기술지식정보서비스_통합정보시스템 고도화 용역.hwp: 276 chunks
  국가철도공단_철도인프라 디지털트윈 정보화전략계획(ISP) 수립 용역(변.hwp: 

In [7]:
# 사용할 청킹 방식 선택
USE_CHUNKS = naive_chunks  # 또는 semantic_chunks
print(f"선택된 청크 수: {len(USE_CHUNKS)}개")

선택된 청크 수: 11339개


## 4. 임베딩 생성 및 벡터 DB 구축

In [8]:
# 임베딩 모델 및 벡터 스토어 초기화
embedding_model = EmbeddingModel(config)
vector_store = VectorStore(config, embedding_model)
vector_store.initialize(collection_name="rfp_documents")

existing_count = vector_store.get_collection_count()
if existing_count > 0:
    print(f"기존 벡터 DB에 {existing_count}개 문서가 있습니다.")
    print("새로 인덱싱하려면 data/vectordb 디렉토리를 삭제 후 다시 실행하세요.")
else:
    print("벡터 DB에 문서를 추가합니다...")
    vector_store.add_documents(USE_CHUNKS)
    print(f"\n최종 저장 문서 수: {vector_store.get_collection_count()}")

기존 벡터 DB에 11567개 문서가 있습니다.
새로 인덱싱하려면 data/vectordb 디렉토리를 삭제 후 다시 실행하세요.


## 5. Retrieval 테스트

다양한 검색 방법 비교: Similarity / MMR / Hybrid

In [9]:
# Retriever 초기화
retriever = Retriever(config, vector_store, embedding_model)

test_query = "국민연금공단이 발주한 이러닝시스템 관련 사업 요구사항을 정리해 줘."
print(f"쿼리: {test_query}\n")
print("=" * 60)

# 1) 유사도 검색
print("\n[1] Similarity Search")
sim_results = retriever.similarity_search(test_query, top_k=3)
for i, r in enumerate(sim_results, 1):
    meta = r.get("metadata", {})
    print(f"  #{i} (score: {r.get('score', 'N/A'):.4f})")
    print(f"      출처: {meta.get('filename', 'N/A')}")
    print(f"      내용: {r['text'][:150]}...")

# 2) MMR 검색
print("\n[2] MMR Search (lambda=0.5)")
mmr_results = retriever.mmr_search(test_query, top_k=3)
for i, r in enumerate(mmr_results, 1):
    meta = r.get("metadata", {})
    print(f"  #{i}")
    print(f"      출처: {meta.get('filename', 'N/A')}")
    print(f"      내용: {r['text'][:150]}...")

# 3) Hybrid 검색
print("\n[3] Hybrid Search (keyword_weight=0.3)")
hybrid_results = retriever.hybrid_search(test_query, top_k=3)
for i, r in enumerate(hybrid_results, 1):
    meta = r.get("metadata", {})
    print(f"  #{i} (hybrid_score: {r.get('hybrid_score', 'N/A'):.4f})")
    print(f"      출처: {meta.get('filename', 'N/A')}")
    print(f"      내용: {r['text'][:150]}...")

쿼리: 국민연금공단이 발주한 이러닝시스템 관련 사업 요구사항을 정리해 줘.


[1] Similarity Search
  #1 (score: 0.6405)
      출처: 국민연금공단_2024년 이러닝시스템 운영 용역.hwp
      내용: 2024년 이러닝시스템 운영 용역 제안요청서
2024년 이러닝시스템 운영 용역 제안요청서
〈 청렴한 업무처리 및 부조리 신고 〉
‘국민연금공단은 모든 계약업무를 투명하고 공정하게 처리하고 있습니다.’
‘계약과정 및 계약이행 시 금품·향응·편의 제공을 요구하는 직원이 있...
  #2 (score: 0.6007)
      출처: 국민연금공단_사업장 사회보험료 지원 고시 개정에 따른 정보시스템 보.hwp
      내용: 사업장 사회보험료 지원 고시 개정에 따른 정보시스템 보완 개발
제안요청서
〈 청렴한 업무처리 및 부조리 신고 안내 〉
‘국민연금공단은 모든 계약업무를 투명하고 공정하게 처리하고 있습니다.’
‘계약과정 및 계약이행 시 금품·향응·편의 제공을 요구하는 직원이 있거나, 지연...
  #3 (score: 0.5800)
      출처: 한국재정정보원_e나라도움 업무시스템 웹 접근성 컨설팅.hwp
      내용: ❍ 인력교체 기간동안 발생하는 업무의 안정화 및 품질저하 방지대책을 제시해야 하고, 인계·인수 기간동안 교체인력에 대한 인건비는 사업자 부담으로 처리하여야 하며, 투입공수에서 제외함
❍ 참여 인원 중 주관기관이 인력교체를 요구하는 경우에는 교체하여야 함
산출정보
착수계...

[2] MMR Search (lambda=0.5)
  #1
      출처: 국민연금공단_2024년 이러닝시스템 운영 용역.hwp
      내용: 2024년 이러닝시스템 운영 용역 제안요청서
2024년 이러닝시스템 운영 용역 제안요청서
〈 청렴한 업무처리 및 부조리 신고 〉
‘국민연금공단은 모든 계약업무를 투명하고 공정하게 처리하고 있습니다.’
‘계약과정 및 계약이행 시 금품·향응·편의 제공을 요구하는 직원이 있...
  #2


## 6. RAG Q&A 시스템

대화 맥락을 유지하며 RFP 문서 기반 Q&A를 수행합니다.

In [10]:
# RAG Generator 초기화
rag = RAGGenerator(config, retriever)

# ── 대화 1: 단일 문서 질문 + 후속 질문 ──
print("=" * 60)
print("대화 1: 단일 문서 질문 + 후속 질문")
print("=" * 60)

q1 = "국민연금공단이 발주한 이러닝시스템 관련 사업 요구사항을 정리해 줘."
print(f"\n[사용자] {q1}\n")
result1 = rag.generate(q1)
print(f"[AI] {result1['answer']}")
print(f"\n응답시간: {result1['elapsed_time']:.2f}s | 검색문서: {len(result1['retrieved_docs'])}개")

q2 = "콘텐츠 개발 관리 요구 사항에 대해서 더 자세히 알려 줘."
print(f"\n{'─'*60}")
print(f"\n[사용자] {q2}\n")
result2 = rag.generate(q2)
print(f"[AI] {result2['answer']}")
print(f"\n응답시간: {result2['elapsed_time']:.2f}s | 검색문서: {len(result2['retrieved_docs'])}개")

대화 1: 단일 문서 질문 + 후속 질문

[사용자] 국민연금공단이 발주한 이러닝시스템 관련 사업 요구사항을 정리해 줘.

[AI] 다음은 국민연금공단이 발주한 이러닝시스템 운영 용역 제안요청서의 핵심 요구사항을 문서 근거 중심으로 간결하게 정리한 내용입니다.

기관/사업 기본
- 발주기관: 국민연금공단
- 사업명: 2024년 이러닝시스템 운영 용역
- 문서 출처: 국민연금공단_2024년 이러닝시스템 운영 용역 제안요청서(또는 동일 명칭의 파일)

핵심 요구사항(카테고리)
- 2-1 교육과정 운영
  - 교육과정 운영 관련 요구사항이 제안요청서의 2-1 항목에 명시되어 있음
  - 구체 내용은 제안요청서 본문에서 확인 필요
- 2-2 이러닝시스템 운영
  - 이러닝시스템 운영과 관련된 요구사항 항목이 2-2로 제시
  - 구체 내용은 제안요청서 본문에서 확인 필요
- 2-3 콘텐츠 개발·관리
  - 콘텐츠 개발 및 관리에 관한 요구사항이 2-3에 제시
  - 구체 내용은 제안요청서 본문에서 확인 필요
- 2-4 개인정보보호 및 정보보안
  - 개인정보처리 및 정보보안 관련 요구사항이 2-4에 명시
  - 구체 내용은 제안요청서 본문에서 확인 필요
- 2-5 기타사항
  - 기타 필요한 요구사항이 2-5에 포함
  - 구체 내용은 제안요청서 본문에서 확인 필요

입찰/제안서 관련 핵심 항목
- 1-1 제안 참가자격
- 1-2 입찰 및 계약 방법
- 1-3 제안서 제출 안내
- 1-4 가격산출
- 2- 제안서 평가
  - 2-1 평가기준 및 평가방법
  - 2-2 제안서 평가 설명회
- Ⅳ 제안서 작성
  - 1. 제안사 준수사항 및 유의사항

제출방식/예산/일정 관련 현황
- 예산: 문서에 예산 규모가 구체적으로 기재되어 있지 않음(확인 필요)
- 제출방식: 제안서 제출 안내가 문서에 포함되어 있음(구체 방식은 본문에서 확인 필요)
- 일정: 본문에 일정 정보가 명시되어 있지 않음(확인 필요)

참고 및 기타
- 청렴/부조리 신고 안내가 문서에 포함되어 있음(청

In [11]:
# ── 대화 2: 다중 문서 비교 ──
rag.reset_memory()
print("=" * 60)
print("대화 2: 다중 문서 비교")
print("=" * 60)

q3 = "고려대학교 차세대 포털 시스템 사업이랑 광주과학기술원의 학사 시스템 기능개선 사업을 비교해 줄래?"
print(f"\n[사용자] {q3}\n")
result3 = rag.generate(q3)
print(f"[AI] {result3['answer']}")
print(f"\n응답시간: {result3['elapsed_time']:.2f}s")

q4 = "고려대학교랑 광주과학기술원 각각 응답 시간에 대한 요구사항이 있나? 문서를 기반으로 정확하게 답변해 줘."
print(f"\n{'─'*60}")
print(f"\n[사용자] {q4}\n")
result4 = rag.generate(q4)
print(f"[AI] {result4['answer']}")
print(f"\n응답시간: {result4['elapsed_time']:.2f}s")

대화 2: 다중 문서 비교

[사용자] 고려대학교 차세대 포털 시스템 사업이랑 광주과학기술원의 학사 시스템 기능개선 사업을 비교해 줄래?

[AI] 아래 비교는 제공된 문서 근거만으로 작성했습니다. 광주과학기술원(학사 시스템 기능개선) 관련 내용은 제공된 문서에 포함되어 있지 않아 문서에서 확인되지 않음을 명시합니다.

1) 기본정보
- 발주기관
  - 고려대학교: 고려대학교 (문서명: 차세대 포털·학사 정보시스템 구축사업)
  - 광주과학기술원: 문서에서 확인되지 않음
- 사업명
  - 고려대학교: 차세대 포털·학사 정보시스템 구축 사업
  - 광주과학기술원: 문서에서 확인되지 않음

2) 사업기간·유지보수·예산
- 사업기간
  - 고려대학교: 계약일로부터 24개월 이내
  - 광주과학기술원: 문서에서 확인되지 않음
- 무상유지보수기간
  - 고려대학교: 사업종료일로부터 12개월
  - 광주과학기술원: 문서에서 확인되지 않음
- 사업예산(금액·지급)
  - 고려대학교: 11,270,000,000원 (부가가치세 포함), 3년 분할지급(2024학년도 약 30%, 2025학년도 약 40%, 2026학년도 약 30%)
  - 광주과학기술원: 문서에서 확인되지 않음

3) 입찰·계약·제출방식
- 입찰 및 계약방법
  - 고려대학교: 제한경쟁입찰(협상에 의한 계약)
  - 광주과학기술원: 문서에서 확인되지 않음
- 제안서 작성/제출 관련
  - 고려대학교: 제안서 목차 및 작성방법 등 지침 문서(본문 및 별첨 참조, 예: 제안서 목차·작성방법 표기)
  - 광주과학기술원: 문서에서 확인되지 않음

4) 사업범위·주요기능(응용SW 개발 기준)
- 고려대학교(주요내용, 문서 근거)
  - 포털시스템: 통합로그인, 통합/지능형 검색, 마이페이지, 공지/알림, 일정관리, 커뮤니티, 게시판, 사용자별 정보서비스, 위젯
  - 연계서비스: 웹메일, 챗봇, 전자결재, 학사/행정 서비스 등 연계
  - 대상: 학생(졸업생 포함), 교직원, 연구원 등 내부 구성원
  - 개인화·접근성

In [12]:
# ── 대화 3: 문서 외 질문 (모른다고 답해야 함) ──
rag.reset_memory()
print("=" * 60)
print("대화 3: 문서 외 질문 (모른다고 답해야 함)")
print("=" * 60)

q5 = "삼성전자가 발주한 반도체 설계 자동화 사업의 요구사항을 알려줘."
print(f"\n[사용자] {q5}\n")
result5 = rag.generate(q5)
print(f"[AI] {result5['answer']}")
print(f"\n응답시간: {result5['elapsed_time']:.2f}s")

대화 3: 문서 외 질문 (모른다고 답해야 함)

[사용자] 삼성전자가 발주한 반도체 설계 자동화 사업의 요구사항을 알려줘.

[AI] - 확인 여부: 문서에서 삼성전자의 반도체 설계 자동화 사업의 요구사항은 확인되지 않음.
- 근거: 본 컨텍스트의 문서는 특정 기관의 예약시스템, 대용량 전송시스템, 모바일오피스 시스템 등과 관련된 요구사항들로 구성되어 있으며, 삼성전자 반도체 설계 자동화 사업에 대한 내용이 포함되어 있지 않음.
- 요약 가능 항목(현재 문서에 포함된 예시들의 형식): 기관/사업명, 요구사항 고유번호 체계(SFR, ECR, STA 등), 시스템 기능 요구사항, 시스템 및 운영 요구사항, 공급 안정성 및 라이선스 요건 등. 다만 삼성 반도체 설계 자동화와 직접 연관된 내용은 없음.
- 다음 단계 제안: 삼성전자 반도체 설계 자동화 사업에 대한 구체 문서나 공고 링크를 제공해 주시면, 그 문서 기반으로 요구사항을 항목별로 정리해 드리겠습니다.

응답시간: 7.55s


In [13]:
# evaluator 모듈 강제 리로드
import importlib
import src.evaluation.evaluator as ev_module
importlib.reload(ev_module)
from src.evaluation.evaluator import RAGEvaluator

# evaluator 재생성
evaluator = RAGEvaluator(config, generator=rag)

# 테스트: 메서드 존재 확인
print("_extract_content 존재:", hasattr(evaluator, "_extract_content"))
print("_clean_json 존재:", hasattr(evaluator, "_clean_json"))
print("eval_models:", config.eval_models)

_extract_content 존재: True
_clean_json 존재: True
eval_models: ['gpt-5-mini', 'gpt-5', 'gpt-5-nano']


In [14]:
from openai import OpenAI

client = OpenAI(api_key=config.openai_api_key)

response = client.chat.completions.create(
    model="gpt-5-mini",
    messages=[{"role": "user", "content": "안녕하세요. 1+1은?"}],
    max_completion_tokens=100,
)

message = response.choices[0].message
print("content:", repr(message.content))
print("type:", type(message.content))
print("전체 message:", message)
print("전체 response:", response)

content: ''
type: <class 'str'>
전체 message: ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)
전체 response: ChatCompletion(id='chatcmpl-DUQylTHy52djGcW36c6eD4nVug6Iu', choices=[Choice(finish_reason='length', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1776145919, model='gpt-5-mini-2025-08-07', object='chat.completion', service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=100, prompt_tokens=15, total_tokens=115, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=100, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))


In [15]:
from openai import OpenAI

client = OpenAI(api_key=config.openai_api_key)

response = client.chat.completions.create(
    model="gpt-5-mini",
    messages=[{"role": "user", "content": '다음 JSON만 반환: {"relevance": 4, "accuracy": 4, "faithfulness": 4, "completeness": 4, "conciseness": 4}'}],
    max_completion_tokens=2000,
    reasoning_effort="low",  # ← 추론 토큰 최소화
)

message = response.choices[0].message
print("content:", repr(message.content))
print("reasoning_tokens:", response.usage.completion_tokens_details.reasoning_tokens)
print("completion_tokens:", response.usage.completion_tokens)

content: '{"relevance": 4, "accuracy": 4, "faithfulness": 4, "completeness": 4, "conciseness": 4}'
reasoning_tokens: 0
completion_tokens: 45


In [16]:
# 1. 리로드
import importlib
import src.evaluation.evaluator as ev_module
importlib.reload(ev_module)
from src.evaluation.evaluator import RAGEvaluator

# 2. evaluator 재생성
evaluator = RAGEvaluator(config, generator=rag)

# 3. 평가 실행
eval_df = evaluator.run_evaluation_suite(use_llm_judge=True)
display(eval_df)


[q1] 국민연금공단이 발주한 이러닝시스템 관련 사업 요구사항을 정리해 줘.
category: single_doc
  [judge] gpt-5-mini 응답: {"relevance": 4, "accuracy": 3, "faithfulness": 3, "completeness": 2, "conciseness": 4}
  [judge] gpt-5 응답: {"relevance": 3, "accuracy": 2, "faithfulness": 2, "completeness": 2, "conciseness": 4}
  [judge] gpt-5-nano 응답: {"relevance": 5, "accuracy": 4, "faithfulness": 4, "completeness": 4, "conciseness": 4}
  elapsed: 24.81s | hit@5: 1.00 | ndcg@5: 1.00
  [follow-up] 콘텐츠 개발 관리 요구 사항에 대해서 더 자세히 알려 줘.
  [judge] gpt-5-mini 응답: {"relevance":4,"accuracy":2,"faithfulness":2,"completeness":3,"conciseness":3}
  [judge] gpt-5 응답: {"relevance": 2, "accuracy": 1, "faithfulness": 1, "completeness": 1, "conciseness": 2}
  [judge] gpt-5-nano 응답: {"relevance": 5, "accuracy": 4, "faithfulness": 4, "completeness": 4, "conciseness": 3}
    elapsed: 31.25s | hit@5: 1.00

[q2] 기초과학연구원 극저온시스템 사업 요구에서 AI 기반 예측에 대한 요구사항이 있나?
category: single_doc
  [judge] gpt-5-mini 응답: {"relevance":5,"accuracy":5,"faithfulness":5,"comp

,id,question,category,answer,elapsed_time,num_retrieved,hit_at_1,hit_at_3,hit_at_5,mrr,...,llm_gpt_5_nano_accuracy,llm_gpt_5_nano_faithfulness,llm_gpt_5_nano_completeness,llm_gpt_5_nano_conciseness,llm_avg_relevance,llm_avg_accuracy,llm_avg_faithfulness,llm_avg_completeness,llm_avg_conciseness,correctly_declined
0,q1,국민연금공단이 발주한 이러닝시스템 관련 사업 요구사항을 정리해 줘.,single_doc,다음은 문서 근거로 정리한 국민연금공단의 이러닝시스템 운영 용역 관련 핵심 요구사항...,24.813187,5,1.0,1.0,1.0,1.0,...,4.0,4.0,4.0,4.0,4.000000,3.000000,3.000000,2.666667,4.000000,NaN
1,q1_followup,콘텐츠 개발 관리 요구 사항에 대해서 더 자세히 알려 줘.,follow_up,"다음은 주어진 컨텍스트에서 확인 가능한 ""콘텐츠 개발 관리"" 관련 요구사항의 정리입...",31.251236,5,1.0,1.0,1.0,1.0,...,4.0,4.0,4.0,3.0,3.666667,2.333333,2.333333,2.666667,2.666667,NaN
2,q2,기초과학연구원 극저온시스템 사업 요구에서 AI 기반 예측에 대한 요구사항이 있나?,single_doc,답변: 문서에서 확인되지 않음.\n\n근거 요약\n- 기초과학연구원 극저온시스템 사...,16.447255,5,1.0,1.0,1.0,1.0,...,4.0,4.0,3.0,4.0,5.000000,4.333333,4.000000,4.000000,4.000000,NaN
3,q2_followup,그럼 모니터링 업무에 대한 요청사항이 있는지 찾아보고 알려 줘.,follow_up,"답변 요약\n- 예, 모니터링 업무에 대한 구체적 요청이 있습니다. 다만 문서별로 ...",27.935426,5,1.0,1.0,1.0,1.0,...,5.0,5.0,4.0,4.0,5.000000,4.666667,4.333333,4.333333,4.000000,NaN
4,q3,"한국 원자력 연구원에서 선량 평가 시스템 고도화 사업을 발주했는데, 이 사업이 왜 ...",single_doc,문서 근거만으로 정리한 추진 목적(요약)\n\n- 규제요건 준수\n - 원자력안전...,15.271968,5,1.0,1.0,1.0,1.0,...,5.0,5.0,4.0,5.0,5.000000,4.333333,4.000000,4.000000,5.000000,NaN
5,q4,고려대학교 차세대 포털 시스템 사업이랑 광주과학기술원의 학사 시스템 기능개선 사업을...,multi_doc,요청하신 비교를 문서 근거만으로 정리합니다. 광주과학기술원(학사 시스템 기능개선)은...,23.058719,5,1.0,1.0,1.0,1.0,...,4.0,4.0,3.0,5.0,4.666667,4.000000,4.000000,3.333333,4.333333,NaN
6,q4_followup,고려대학교랑 광주과학기술원 각각 응답 시간에 대한 요구사항이 있나? 문서를 기반으로...,follow_up,요청에 따라 문서 근거만으로 비교 정리합니다.\n\n- 고려대학교(차세대 포털·학사...,20.095282,5,1.0,1.0,1.0,1.0,...,3.0,4.0,3.0,4.0,5.000000,3.333333,3.333333,4.000000,4.333333,NaN
7,q5,교육이나 학습 관련해서 다른 기관이 발주한 사업은 없나?,cross_doc,다른 기관이 발주한 교육/학습 관련 사업은 아래와 같이 확인됩니다.\n\n- 기관:...,41.721059,5,1.0,1.0,1.0,1.0,...,4.0,4.0,4.0,3.0,4.666667,2.333333,2.333333,3.000000,3.333333,NaN
8,q6,삼성전자가 발주한 반도체 설계 자동화 사업의 요구사항을 알려줘.,out_of_scope,문서에서 확인되지 않음.\n\n제공된 컨텍스트에는 삼성전자 발주의 반도체 설계 자동...,11.302800,5,0.0,0.0,0.0,0.0,...,4.0,4.0,3.0,5.0,4.333333,4.666667,4.666667,3.000000,4.666667,True


## 7. 성능 평가

LLM-as-a-Judge 방식으로 답변 품질 자동 평가

**평가 지표:** Relevance, Accuracy, Faithfulness, Completeness, Conciseness (각 1-5점), Keyword Recall, Response Time

In [17]:
# 평가 실행
evaluator = RAGEvaluator(config, generator=rag)

print("전체 평가 세트 실행 중...")
print("(각 질문에 대해 RAG 답변 생성 + LLM Judge 평가)\n")
eval_df = evaluator.run_evaluation_suite(use_llm_judge=True)

display(eval_df)

전체 평가 세트 실행 중...
(각 질문에 대해 RAG 답변 생성 + LLM Judge 평가)


[q1] 국민연금공단이 발주한 이러닝시스템 관련 사업 요구사항을 정리해 줘.
category: single_doc
  [judge] gpt-5-mini 응답: {"relevance": 4, "accuracy": 3, "faithfulness": 2, "completeness": 2, "conciseness": 4}
  [judge] gpt-5 응답: {"relevance": 4, "accuracy": 3, "faithfulness": 3, "completeness": 2, "conciseness": 4}
  [judge] gpt-5-nano 응답: {"relevance": 4, "accuracy": 3, "faithfulness": 3, "completeness": 3, "conciseness": 4}
  elapsed: 28.46s | hit@5: 1.00 | ndcg@5: 1.00
  [follow-up] 콘텐츠 개발 관리 요구 사항에 대해서 더 자세히 알려 줘.
  [judge] gpt-5-mini 응답: {"relevance":5,"accuracy":3,"faithfulness":3,"completeness":3,"conciseness":5}
  [judge] gpt-5 응답: {"relevance": 4, "accuracy": 2, "faithfulness": 1, "completeness": 3, "conciseness": 4}
  [judge] gpt-5-nano 응답: {"relevance": 4, "accuracy": 4, "faithfulness": 4, "completeness": 4, "conciseness": 4}
    elapsed: 24.07s | hit@5: 1.00

[q2] 기초과학연구원 극저온시스템 사업 요구에서 AI 기반 예측에 대한 요구사항이 있나?
category: single_doc
  [judge] gpt-5-mini 

,id,question,category,answer,elapsed_time,num_retrieved,hit_at_1,hit_at_3,hit_at_5,mrr,...,llm_gpt_5_nano_accuracy,llm_gpt_5_nano_faithfulness,llm_gpt_5_nano_completeness,llm_gpt_5_nano_conciseness,llm_avg_relevance,llm_avg_accuracy,llm_avg_faithfulness,llm_avg_completeness,llm_avg_conciseness,correctly_declined
0,q1,국민연금공단이 발주한 이러닝시스템 관련 사업 요구사항을 정리해 줘.,single_doc,다음은 제시된 국민연금공단의 이러닝시스템 운영 용역 제안요청서(2024년) 기반의 ...,28.464842,5,1.0,1.0,1.0,1.0,...,3.0,3.0,3.0,4.0,4.000000,3.000000,2.666667,2.333333,4.000000,NaN
1,q1_followup,콘텐츠 개발 관리 요구 사항에 대해서 더 자세히 알려 줘.,follow_up,다음은 제안요청서상 콘텐츠 개발·관리(Content Development & Man...,24.069952,5,1.0,1.0,1.0,1.0,...,4.0,4.0,4.0,4.0,4.333333,3.000000,2.666667,3.333333,4.333333,NaN
2,q2,기초과학연구원 극저온시스템 사업 요구에서 AI 기반 예측에 대한 요구사항이 있나?,single_doc,- 결론: 문서에서 “AI 기반 예측”에 대한 구체적 요구사항은 확인되지 않음.\n...,14.733535,5,1.0,1.0,1.0,1.0,...,4.0,4.0,3.0,4.0,3.666667,3.333333,3.333333,3.000000,4.000000,NaN
3,q2_followup,그럼 모니터링 업무에 대한 요청사항이 있는지 찾아보고 알려 줘.,follow_up,요약\n- 예: 모니터링 관련 업무 요구가 HDIMS 재구축(3단계) 문서에 명시되...,35.335939,5,0.0,1.0,1.0,0.5,...,4.0,4.0,4.0,4.0,4.666667,4.000000,3.333333,4.000000,3.666667,NaN
4,q3,"한국 원자력 연구원에서 선량 평가 시스템 고도화 사업을 발주했는데, 이 사업이 왜 ...",single_doc,간단 정리 — 추진 목적 (문서 근거 중심)\n\n발주 기관: 한국원자력연구원 \...,17.834909,5,1.0,1.0,1.0,1.0,...,5.0,5.0,5.0,4.0,5.000000,4.666667,4.333333,5.000000,4.666667,NaN
5,q4,고려대학교 차세대 포털 시스템 사업이랑 광주과학기술원의 학사 시스템 기능개선 사업을...,multi_doc,요청하신 비교는 문서 근거 기반으로만 작성합니다. 광주과학기술원 관련 내용은 제공된...,24.901321,5,1.0,1.0,1.0,1.0,...,4.0,4.0,4.0,3.0,5.000000,4.333333,4.000000,4.000000,3.333333,NaN
6,q4_followup,고려대학교랑 광주과학기술원 각각 응답 시간에 대한 요구사항이 있나? 문서를 기반으로...,follow_up,비교(문서 근거 기반)\n\n- 광주과학기술원 — 실시간통합연구비관리시스템(RCMS...,19.856622,5,1.0,1.0,1.0,1.0,...,5.0,4.0,4.0,4.0,4.666667,4.000000,3.666667,4.333333,4.666667,NaN
7,q5,교육이나 학습 관련해서 다른 기관이 발주한 사업은 없나?,cross_doc,"예, 교육/학습 관련 발주 사업은 다수의 다른 기관에서 진행되고 있습니다. 각 기관...",40.593082,5,1.0,1.0,1.0,1.0,...,3.0,4.0,4.0,3.0,5.000000,2.333333,2.333333,4.000000,3.333333,NaN
8,q6,삼성전자가 발주한 반도체 설계 자동화 사업의 요구사항을 알려줘.,out_of_scope,문서에서 확인되지 않음\n\n- 제공된 컨텍스트에는 삼성전자의 반도체 설계 자동화 ...,21.725321,5,0.0,0.0,0.0,0.0,...,1.0,1.0,1.0,5.0,3.666667,3.000000,2.666667,3.333333,4.333333,True


In [18]:
# 평가 결과 요약
report = evaluator.summary_report(eval_df)

print("=" * 60)
print("평가 결과 요약")
print("=" * 60)
print(f"\n총 질문 수: {report['total_questions']}")
print(f"평균 응답 시간: {report['avg_elapsed_time']:.2f}s")
print(f"최대 응답 시간: {report['max_elapsed_time']:.2f}s")

if "avg_keyword_recall" in report:
    print(f"평균 키워드 재현율: {report['avg_keyword_recall']:.1%}")

llm_keys = [k for k in report if k.startswith("avg_llm_")]
if llm_keys:
    print(f"\nLLM Judge 점수 (1-5점):")
    for k in llm_keys:
        name = k.replace("avg_llm_", "").title()
        print(f"  {name}: {report[k]:.2f}")

print(f"\n카테고리별 성능:")
for cat, cat_report in report.get("by_category", {}).items():
    print(f"  [{cat}] 질문 {cat_report['count']}개, 평균 {cat_report['avg_elapsed_time']:.2f}s")

evaluator.save_results()

평가 결과 요약

총 질문 수: 9
평균 응답 시간: 25.28s
최대 응답 시간: 40.59s
평균 키워드 재현율: 87.5%

LLM Judge 점수 (1-5점):
  Gpt_5_Mini_Relevance: 4.89
  Gpt_5_Mini_Accuracy: 3.67
  Gpt_5_Mini_Faithfulness: 3.44
  Gpt_5_Mini_Completeness: 3.89
  Gpt_5_Mini_Conciseness: 4.44
  Gpt_5_Relevance: 4.33
  Gpt_5_Accuracy: 3.22
  Gpt_5_Faithfulness: 2.56
  Gpt_5_Completeness: 3.67
  Gpt_5_Conciseness: 3.78
  Gpt_5_Nano_Relevance: 4.11
  Gpt_5_Nano_Accuracy: 3.67
  Gpt_5_Nano_Faithfulness: 3.67
  Gpt_5_Nano_Completeness: 3.56
  Gpt_5_Nano_Conciseness: 3.89
  Avg_Relevance: 4.44
  Avg_Accuracy: 3.52
  Avg_Faithfulness: 3.22
  Avg_Completeness: 3.70
  Avg_Conciseness: 4.04

카테고리별 성능:
  [single_doc] 질문 3개, 평균 20.34s
  [follow_up] 질문 3개, 평균 26.42s
  [multi_doc] 질문 1개, 평균 24.90s
  [cross_doc] 질문 1개, 평균 40.59s
  [out_of_scope] 질문 1개, 평균 21.73s
saved: evaluation/eval_results.json


## 8. 실험 비교

다양한 Retrieval 설정 조합을 비교합니다.

In [19]:
import time

experiments = [
    {"name": "baseline_sim_k3", "method": "similarity", "top_k": 3},
    {"name": "baseline_sim_k5", "method": "similarity", "top_k": 5},
    {"name": "baseline_sim_k10", "method": "similarity", "top_k": 10},
    {"name": "mmr_k5", "method": "mmr", "top_k": 5},
    {"name": "hybrid_k5", "method": "hybrid", "top_k": 5},
]

test_queries = [
    "국민연금공단이 발주한 이러닝시스템 관련 사업 요구사항을 정리해 줘.",
    "한국 원자력 연구원에서 선량 평가 시스템 고도화 사업을 발주했는데, 이 사업이 왜 추진되는지 목적을 알려 줘.",
    "교육이나 학습 관련해서 다른 기관이 발주한 사업은 없나?",
]

experiment_results = []

for exp in experiments:
    print(f"\n{'='*50}")
    print(f"실험: {exp['name']}")
    
    exp_times = []
    for query in test_queries:
        rag.reset_memory()
        start = time.time()
        result = rag.generate(query, retrieval_method=exp["method"], top_k=exp["top_k"])
        elapsed = time.time() - start
        exp_times.append(elapsed)
        print(f"  {query[:40]}... -> {elapsed:.2f}s")
    
    experiment_results.append({
        "experiment": exp["name"],
        "method": exp["method"],
        "top_k": exp["top_k"],
        "avg_time": sum(exp_times) / len(exp_times),
        "max_time": max(exp_times),
    })

exp_df = pd.DataFrame(experiment_results)
print(f"\n{'='*50}")
print("실험 결과 비교")
display(exp_df)


실험: baseline_sim_k3
  국민연금공단이 발주한 이러닝시스템 관련 사업 요구사항을 정리해 줘.... -> 37.41s
  한국 원자력 연구원에서 선량 평가 시스템 고도화 사업을 발주했는데, 이 ... -> 11.33s
  교육이나 학습 관련해서 다른 기관이 발주한 사업은 없나?... -> 35.86s

실험: baseline_sim_k5
  국민연금공단이 발주한 이러닝시스템 관련 사업 요구사항을 정리해 줘.... -> 24.92s
  한국 원자력 연구원에서 선량 평가 시스템 고도화 사업을 발주했는데, 이 ... -> 10.82s
  교육이나 학습 관련해서 다른 기관이 발주한 사업은 없나?... -> 33.63s

실험: baseline_sim_k10
  국민연금공단이 발주한 이러닝시스템 관련 사업 요구사항을 정리해 줘.... -> 25.78s
  한국 원자력 연구원에서 선량 평가 시스템 고도화 사업을 발주했는데, 이 ... -> 15.67s
  교육이나 학습 관련해서 다른 기관이 발주한 사업은 없나?... -> 26.09s

실험: mmr_k5
  국민연금공단이 발주한 이러닝시스템 관련 사업 요구사항을 정리해 줘.... -> 27.33s
  한국 원자력 연구원에서 선량 평가 시스템 고도화 사업을 발주했는데, 이 ... -> 10.43s
  교육이나 학습 관련해서 다른 기관이 발주한 사업은 없나?... -> 20.63s

실험: hybrid_k5
  국민연금공단이 발주한 이러닝시스템 관련 사업 요구사항을 정리해 줘.... -> 25.71s
  한국 원자력 연구원에서 선량 평가 시스템 고도화 사업을 발주했는데, 이 ... -> 9.49s
  교육이나 학습 관련해서 다른 기관이 발주한 사업은 없나?... -> 18.36s

실험 결과 비교


,experiment,method,top_k,avg_time,max_time
0,baseline_sim_k3,similarity,3,28.200628,37.406666
1,baseline_sim_k5,similarity,5,23.126219,33.633889
2,baseline_sim_k10,similarity,10,22.513210,26.090652
3,mmr_k5,mmr,5,19.467340,27.334864
4,hybrid_k5,hybrid,5,17.855577,25.713422


## 9. 인터랙티브 Q&A (자유 질문)

In [20]:
# 대화 초기화 후 자유 질문
rag.reset_memory()

question = "기초과학연구원 극저온시스템 사업 요구에서 AI 기반 예측에 대한 요구사항이 있나?"

result = rag.generate(question, stream=True)
print(f"\n\n응답시간: {result['elapsed_time']:.2f}s")
print(f"참조 문서:")
for i, doc in enumerate(result["retrieved_docs"], 1):
    meta = doc.get("metadata", {})
    print(f"  {i}. {meta.get('filename', 'N/A')} (score: {doc.get('score', 'N/A'):.4f})")



응답시간: 10.51s
참조 문서:
  1. 서울특별시교육청_서울특별시교육청 지능정보화전략계획(ISP) 수립(2차) .hwp (score: 0.5097)
  2. 전북대학교_JST 공유대학(원) xAPI기반 LRS시스템 구축.hwp (score: 0.5060)
  3. 그랜드코리아레저(주)_2024년도 GKL  그룹웨어 시스템 구축 용역.hwp (score: 0.5060)
  4. 한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp (score: 0.5056)
  5. 인천광역시_인천일자리플랫폼 정보시스템 구축 ISP 수립용역.hwp (score: 0.4991)


In [21]:
# 후속 질문 (대화 맥락 유지 - reset_memory() 호출하지 않음)
follow_up = "그럼 모니터링 업무에 대한 요청사항이 있는지 찾아보고 알려 줘."

result = rag.generate(follow_up, stream=True)
print(f"\n\n응답시간: {result['elapsed_time']:.2f}s")



응답시간: 25.73s


In [25]:
# 1. 새 질문지 임포트
from src.evaluation.multi_dataset import EVALUATION_QUESTIONS as MULTI_QUESTIONS

print(f"질문 수: {len(MULTI_QUESTIONS)}개")
print(f"첫 번째 질문: {MULTI_QUESTIONS[0]['question']}")

질문 수: 50개
첫 번째 질문: 한국생산기술연구원의 EIP3.0 고압가스 안전관리 시스템 구축 용역이랑 광주과학기술원의 RCMS 연계 모듈 변경 사업의 핵심 요구사항을 비교해 줄래?


In [ ]:
# 2. evaluator 리로드 (혹시 안 했다면)
import importlib
import src.evaluation.evaluator as ev_module
importlib.reload(ev_module)
from src.evaluation.evaluator import RAGEvaluator

# 3. evaluator 재생성
evaluator = RAGEvaluator(config, generator=rag)

# 4. 새 질문지로 평가 실행
print("멀티 질문지 평가 시작...")
eval_df = evaluator.run_evaluation_suite(
    questions=MULTI_QUESTIONS,   # ← 여기만 바꾸면 됩니다
    use_llm_judge=True
)

display(eval_df)
evaluator.save_results("evaluation/eval_results.json")

멀티 질문지 평가 시작...

[q1] 한국생산기술연구원의 EIP3.0 고압가스 안전관리 시스템 구축 용역이랑 광주과학기술원의 RCMS 연계 모듈 변경 사업의 핵심 요구사항을 비교해 줄래?
category: multi_doc
  [judge] gpt-5-mini 응답: {"relevance":5,"accuracy":4,"faithfulness":5,"completeness":4,"conciseness":4}
  [judge] gpt-5 응답: {"relevance": 4, "accuracy": 3, "faithfulness": 3, "completeness": 3, "conciseness": 3}
  [judge] gpt-5-nano 응답: {"relevance":4,"accuracy":4,"faithfulness":4,"completeness":3,"conciseness":5}
  elapsed: 24.42s | hit@5: 1.00 | ndcg@5: 0.99
  [follow-up] 두 사업에서 품질보증이나 시험운영 관련 요구가 문서에 명시되어 있어?
  [judge] gpt-5-mini 응답: {"relevance":5,"accuracy":5,"faithfulness":5,"completeness":4,"conciseness":4}
  [judge] gpt-5 응답: {"relevance":4,"accuracy":3,"faithfulness":2,"completeness":3,"conciseness":3}
  [judge] gpt-5-nano 응답: {"relevance":4,"accuracy":4,"faithfulness":4,"completeness":3,"conciseness":5}
    elapsed: 38.14s | hit@5: 1.00

[q2] 대한장애인체육회 전국장애인체육대회 홈페이지 사업과 대검찰청 APC-HUB 홈페이지 사업의 웹 표준·호환성 요구사항과 성능 요구사항을 비교해 줄래?
category: multi_doc
  [judge]

In [24]:
evaluator.save_results("evaluation/eval_results.json")

saved: evaluation/eval_results.json
